# IRRM-CODEC: run and analyze

This notebook shows how to train from two inputs: an AIRR table and a parquet file with TCRemP embeddings.

## 1. Define inputs

The AIRR file must contain `clone_id`, `junction_aa`, `v_call`, `j_call`, `locus`.
The embeddings parquet must contain `clone_id` plus either `tcremp_emb` or numeric embedding columns.

In the model pipeline, CDR3 sequences are gap-padded to a fixed length of 40 amino acids before tokenization.

In [1]:
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
airr_path = '/projects/immunestatus/vdjdb_olga/airr_format/trb_background_100k.tsv' #root / 'data' / 'sample_airr.tsv'
embeddings_path = '/projects/immunestatus/vdjdb_olga/tcremp/trb_background_100k_embeddings.parquet'
output_dir = root / 'artifacts' / 'forward_demo_trb'

airr_path, embeddings_path, output_dir

('/projects/immunestatus/vdjdb_olga/airr_format/trb_background_100k.tsv',
 '/projects/immunestatus/vdjdb_olga/tcremp/trb_background_100k_embeddings.parquet',
 PosixPath('/home/evlasova/irrm-codec/artifacts/forward_demo_trb'))

In [2]:
import sys
sys.path.insert(0, str(root))

## 2. Preview the two files

In [3]:
import pandas as pd

airr_df = pd.read_csv(airr_path, sep='\t')
emb_df = pd.read_parquet(embeddings_path)

display(airr_df.head())
display(emb_df.head())

,junction_aa,v_call,j_call,locus
0,CATSDPGTGHQPQHF,TRBV24-1,TRBJ1-5,beta
1,CAISEGLGQGETQYF,TRBV10-3,TRBJ2-5,beta
2,CASSQVGTGVYEQYF,TRBV3-1,TRBJ2-7,beta
3,CASSLGADTQYF,TRBV13,TRBJ2-3,beta
4,CASSRTGNEQYF,TRBV27,TRBJ2-7,beta


,0_b_v,0_b_j,0_b_cdr3,1_b_v,1_b_j,1_b_cdr3,2_b_v,2_b_j,2_b_cdr3,3_b_v,...,2996_b_cdr3,2997_b_v,2997_b_j,2997_b_cdr3,2998_b_v,2998_b_j,2998_b_cdr3,2999_b_v,2999_b_j,2999_b_cdr3
0,510.0,0.0,1070.0,444.0,221.0,1200.0,776.0,0.0,890.0,776.0,...,1100.0,697.0,221.0,980.0,682.0,196.0,1320.0,488.0,196.0,890.0
1,398.0,223.0,1410.0,464.0,198.0,1120.0,782.0,223.0,1070.0,782.0,...,720.0,727.0,198.0,1040.0,686.0,197.0,1460.0,428.0,197.0,1070.0
2,775.0,221.0,1330.0,709.0,0.0,820.0,423.0,221.0,990.0,423.0,...,1160.0,670.0,0.0,680.0,615.0,107.0,1080.0,765.0,107.0,730.0
3,765.0,212.0,1460.0,761.0,213.0,730.0,709.0,212.0,1200.0,709.0,...,850.0,434.0,213.0,770.0,453.0,202.0,1090.0,763.0,202.0,580.0
4,376.0,221.0,1400.0,0.0,0.0,750.0,768.0,221.0,1120.0,768.0,...,1170.0,689.0,0.0,730.0,712.0,107.0,1210.0,492.0,107.0,500.0


## 3. Launch training

In [ ]:
import subprocess

cmd = [
    sys.executable, "-m", "irrm_codec.train_forward",
    "--airr-path", str(airr_path),
    "--embeddings-path", str(embeddings_path),
    "--output-dir", str(output_dir),
    "--locus", "beta",
    "--max-len", "40",
    "--epochs", "40",
    "--lr", "3e-4",
    "--weight-decay", "1e-3",
    "--dropout", "0.2",
    "--encoder-type", "plain_conv",
    "--hidden-dim", "192",
    "--mlp-dim", "512",
    "--mlp-hidden-dim", "1024",
    "--dilations", "1,2,4,8",
    "--early-stopping-patience", "5",
    "--scheduler", "plateau",
    "--scheduler-patience", "2",
    "--scheduler-factor", "0.5",
]


subprocess.run(cmd, cwd=root, check=True)
cmd

2026-04-02 16:24:45,606 | INFO | starting forward training
2026-04-02 16:24:45,636 | INFO | output_dir=/home/evlasova/irrm-codec/artifacts/forward_demo_trb
2026-04-02 16:24:45,636 | INFO | device=cuda seed=42
2026-04-02 16:24:45,637 | INFO | hyperparameters batch_size=256 epochs=40 lr=0.000300 weight_decay=0.001000 max_len=40 encoder_type=plain_conv hidden_dim=192 mlp_dim=512 mlp_hidden_dim=1024 dropout=0.200 dilations=1,2,4,8 num_workers=0 log_interval=10 early_stopping_patience=5 scheduler=plateau
2026-04-02 16:25:20,116 | INFO | loaded data total=100000 train=80000 val=10000 test=10000 embedding_dim=9000
2026-04-02 16:25:20,116 | INFO | dataloader batches train=313 val=40 test=40
2026-04-02 16:25:20,116 | INFO | model parameters total=10406184 trainable=10406184
2026-04-02 16:25:20,122 | INFO | epoch 1/40 started
2026-04-02 16:25:27,596 | INFO | saved checkpoint path=/home/evlasova/irrm-codec/artifacts/forward_demo_trb/last.pt
2026-04-02 16:25:27,826 | INFO | new best checkpoint pat

## 4. Inspect saved metrics and merge stats

In [ ]:
import json
import pandas as pd

history = json.loads((output_dir / 'history.json').read_text())
test_metrics = json.loads((output_dir / 'test_metrics.json').read_text())
data_stats = json.loads((output_dir / 'data_stats.json').read_text())
history_df = pd.json_normalize(history)

display(history_df)
display(test_metrics)
display(data_stats)

In [ ]:
history_df.plot(x='epoch', y=['train.loss', 'val.loss'], figsize=(8, 4), grid=True, title='Loss by epoch')

## 5. Load a checkpoint

In [ ]:
import torch
from irrm_codec.forward_model import ForwardModel

checkpoint = torch.load(output_dir / "best.pt", map_location="cpu")

extra = checkpoint["extra"]
model = ForwardModel(
    output_dim=extra["embedding_dim"],
    max_len=extra.get("max_len", 40),
    hidden_dim=extra.get("hidden_dim", 192),
    mlp_dim=extra.get("mlp_dim", 512),
    mlp_hidden_dim=extra.get("mlp_hidden_dim", 1024),
    dropout=extra.get("dropout", 0.2),
    dilations=tuple(int(x) for x in str(extra.get("dilations", "1,2,4,8")).split(",")),
    encoder_type=extra.get("encoder_type", "residual"),
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

checkpoint["metrics"]


In [ ]:
model.predict('CASSLAPGATNEKLFF')